# Can we predict a warranty claim within a year?
Protocol: `protocol.yaml`. Holdout rows fixed by `holdout.csv`; 5-fold stratified CV on training rows; model chosen by mean CV ROC AUC; holdout scored once.

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx) == 1400 and len(set(hold_idx)) == 1400
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
target = 'claim_within_year'
# device_serial excluded (identifier); remaining columns are all known at sale/registration time
cat, num = ['channel'], ['device_age_years','price_eur','daily_hours']
features = cat + num
X, y = df[features], df[target]
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
Xho, yho = X.iloc[hold_idx], y.iloc[hold_idx]
print(len(train_idx), len(hold_idx), ytr.mean(), yho.mean())

## Candidates (named before any scoring)
Preprocessing lives inside each pipeline, so it is fitted within CV folds.

In [ ]:
def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
candidates = {
 'baseline_prior': make_pipeline(pre(False), DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(pre(True), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.1': make_pipeline(pre(True), LogisticRegression(C=0.1, max_iter=1000)),
 'random_forest': make_pipeline(pre(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
 'hist_gb': make_pipeline(pre(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
print(list(candidates))

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds = [(tr.tolist(), va.tolist()) for tr, va in skf.split(Xtr, ytr)]
cv_scores = {}
for name, m in candidates.items():
    s = []
    for tr, va in folds:
        mm = clone(m).fit(Xtr.iloc[tr], ytr.iloc[tr])
        s.append(roc_auc_score(ytr.iloc[va], mm.predict_proba(Xtr.iloc[va])[:,1]))
    cv_scores[name] = s
cv_table = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(cv_table.sort_values('mean', ascending=False))
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)

## Holdout: refit chosen candidate on all training rows, score once

In [ ]:
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = roc_auc_score(yho, final.predict_proba(Xho)[:,1])
print('holdout roc_auc:', holdout_auc)

In [ ]:
pos = lambda idx: [int(i) for i in idx]
evidence = {
 'features': features,
 'candidates': list(candidates),
 'cv_folds': [[pos(train_idx[tr]), pos(train_idx[va])] for tr, va in folds],
 'cv_scores': {k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': pos(hold_idx),
 'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json','w'))